In [1]:
import sys
sys.path.append("../Src")

import yaml
import numpy as np
import pandas as pd
from q3_marine_life_functions import *

In [2]:
with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)

In [3]:
species = pd.read_parquet(config["output_data"]["file9"])
microplastics = pd.read_parquet(config["output_data"]["file3"])

In [4]:
species_clean = clean_species(species)
microplastics = clean_microplastics(microplastics)

Species: 10,412 → 10,205 rows after dropping missing coords
Dropped: 207 rows (2.0%)
Microplastics concentration_class after cleaning:
concentration_class
Very Low      5950
Low           2516
Medium       10664
High          2642
Very High      758
Name: count, dtype: int64
Unexpected values: 0


In [5]:
species_ml = spatial_join_species_mp(species_clean, microplastics)

Matched within 50 km : 6,652 / 10,205 (65.2%)
concentration_class
Very Low     2707
Low           588
Medium       2954
High          230
Very High     173
Name: count, dtype: int64


In [6]:
ingestion_df = ingestion_rate_by_class(species_ml)
stat, p = run_kruskal_wallis(species_ml)
pairwise_df = run_pairwise_mannwhitney(species_ml)

concentration_class  n_animals  n_with_ingestion  ingestion_rate
           Very Low       2707               199        7.351311
                Low        588                62       10.544218
             Medium       2954               403       13.642519
               High        230                62       26.956522
          Very High        173                12        6.936416
Kruskal-Wallis H = 119.762,  p = 0.0000
→ Significant difference across classes (p < 0.05)
Bonferroni threshold: 0.0050

Very Low vs Low: p_adj=0.0938  —
Very Low vs Medium: p_adj=0.0000  ✓
Very Low vs High: p_adj=0.0000  ✓
Very Low vs Very High: p_adj=1.0000  —
Low vs Medium: p_adj=0.4222  —
Low vs High: p_adj=0.0000  ✓
Low vs Very High: p_adj=1.0000  —
Medium vs High: p_adj=0.0000  ✓
Medium vs Very High: p_adj=0.1152  —
High vs Very High: p_adj=0.0000  ✓


In [7]:
fig_h1 = plot_ingestion_by_class(ingestion_df)
fig_h1.show()

In [8]:
chi2, p, odds_ratio = run_ghost_net_chi2(species)

Chi-square: 27.319,  p = 0.0000,  df = 1
Odds ratio: 1.423

Entanglement rate — Small: 8.2%,  Large: 11.2%


In [9]:
fig_h2 = plot_ghost_net(chi2, p, odds_ratio)
fig_h2.show()

In [10]:
profile = build_plastic_profile(species)
fig_h3 = plot_plastic_heatmap(profile)
fig_h3.show()

In [11]:
df_enc, encoders = encode_features(species_ml)
X_train, X_test, y_train, y_test = split_data(df_enc)

rf_base = train_baseline_rf(X_train, y_train)
evaluate_model(rf_base, X_test, y_test, "Baseline Random Forest")

rf_search = tune_rf(X_train, y_train)
rf_tuned = rf_search.best_estimator_
evaluate_model(rf_tuned, X_test, y_test, "Tuned Random Forest")

gb = train_gradient_boosting(X_train, y_train)
evaluate_model(gb, X_test, y_test, "Gradient Boosting")

Train: 8,164  |  Test: 2,041
Positive rate — train: 13.1%  |  test: 13.1%

=== Baseline Random Forest ===
              precision    recall  f1-score   support

No ingestion       0.94      0.89      0.92      1774
   Ingestion       0.48      0.65      0.55       267

    accuracy                           0.86      2041
   macro avg       0.71      0.77      0.73      2041
weighted avg       0.88      0.86      0.87      2041

ROC-AUC : 0.836
Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best params : {'max_depth': 10, 'max_features': 'log2', 'min_samples_split': 18, 'n_estimators': 149}
Best CV AUC : 0.847

=== Tuned Random Forest ===
              precision    recall  f1-score   support

No ingestion       0.96      0.86      0.91      1774
   Ingestion       0.44      0.74      0.55       267

    accuracy                           0.84      2041
   macro avg       0.70      0.80      0.73      2041
weighted avg       0.89      0.84      0.86      2041

ROC-AUC : 0

0.8522847708684325

In [12]:
plot_feature_importance(rf_tuned).show()
plot_roc_curve(rf_tuned, X_test, y_test).show()
plot_probability_distribution(rf_tuned, X_test, y_test).show()